In [ ]:
import cv2
import json
import time
import os
from kafka import KafkaProducer
from PIL import Image

# --- 1. Настройки ---
VIDEO_PATH = "/home/jovyan/dataset/sea_video.mp4" 
OUTPUT_DIR = "/home/jovyan/dataset/Detection/images/val/"
TOPIC_NAME = 'marine_stream'

os.makedirs(OUTPUT_DIR, exist_ok=True)

producer = KafkaProducer(
    bootstrap_servers=['kafka:29092'],
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

cap = cv2.VideoCapture(VIDEO_PATH)
frame_count = 0

print("Kafka запущен")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            continue

        # Возвращаем каждый 15-й кадр для плавности
        if frame_count % 15 == 0:
            file_name = f"frame_{frame_count}.jpg"
            full_path = os.path.join(OUTPUT_DIR, file_name)
            
            # Сохранение через PIL
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img_pil = Image.fromarray(frame_rgb)
            
            temp_path = full_path + ".tmp"
            img_pil.save(temp_path, "JPEG", quality=85)
            os.replace(temp_path, full_path)

            message = {"event_id": frame_count, "filename": file_name, "timestamp": time.time()}
            producer.send(TOPIC_NAME, value=message)
            
            time.sleep(0.01)

        frame_count += 1

except KeyboardInterrupt:
    print("🛑 Остановлено")
finally:
    cap.release()
    producer.close()

Kafka запущен
